In [ ]:
from resources.imports import *

## Gaussian Process NN

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
def gaussian_mixture_model(inputs, num_components=3):
    # Assuming inputs have 2 dimensions (mean and log_variance)
    mean, log_variance = tf.split(inputs, 2, axis=-1)

    # Calculate variance from log_variance
    variance = tf.exp(log_variance)

    # Create a Lambda layer to sample from the Gaussian distribution
    def sampling(args):
        mean, variance = args
        batch_size = K.shape(mean)[0]
        dim = K.int_shape(mean)[1]
        epsilon = K.random_normal(shape=(batch_size, dim))
        return mean + K.sqrt(variance) * epsilon

    samples = Lambda(sampling, output_shape=(num_components,), name='sample')([mean, variance])

    # Create a mixture density network output layer
    outputs = Dense(num_components * 3, activation='linear')(inputs)

    # Reshape the output to separate means, variances, and weights for each component
    outputs = Reshape((num_components, 3))(outputs)

    model = Model(inputs, outputs, name='gaussian_mixture_model')
    return model

# Create a simple example model
input_dim = 10  # Adjust based on your input dimensions
num_components = 3
inputs = Input(shape=(input_dim,), name='input')
outputs = gaussian_mixture_model(inputs, num_components=num_components)

# Compile the model
optimizer = tf.keras.optimizers.Adam(lr=0.001)
outputs.compile(optimizer=optimizer, loss='mean_squared_error')  # Adjust the loss function as needed

# Print the model summary
outputs.summary()


NameError: name 'Reshape' is not defined

In [ ]:
## Gaussian Process Neural Network (GPNN)

import torch
import torch.nn as nn
import torch.optim as optim
from gpytorch.models import ExactGP
from gpytorch.kernels import RBFKernel, ScaleKernel
from gpytorch.means import ConstantMean
from gpytorch.likelihoods import GaussianLikelihood
import gpytorch

# Neural Network Feature Extractor
class FeatureNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)  # Latent feature representation

# Gaussian Process Model
class GPModel(ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(GPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = ConstantMean()
        self.covar_module = ScaleKernel(RBFKernel())  # Gaussian kernel (RBF)

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# Example usage
input_dim = 2  # Assume (Δx, Δy) for each node
hidden_dim = 32
output_dim = 16  # GP operates on 16D latent space

feature_nn = FeatureNN(input_dim, hidden_dim, output_dim)

# Simulated Training Data
train_x = torch.randn(100, 2)  # 100 nodes, 2D input
train_y = torch.randn(100)  # Target stresses

# Transform inputs using NN
latent_x = feature_nn(train_x)

# GP Model
likelihood = GaussianLikelihood()
gp_model = GPModel(latent_x, train_y, likelihood)

# Training the GP
gp_model.train()
likelihood.train()
optimizer = torch.optim.Adam([
    {'params': gp_model.parameters()},  
], lr=0.01)


In [ ]:
# RBF Neural Network

import torch
import torch.nn as nn
import torch.optim as optim

# Gaussian (RBF) Activation Function
class RBF_Layer(nn.Module):
    def __init__(self, num_centers, input_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_centers, input_dim))
        self.beta = nn.Parameter(torch.ones(num_centers) * 0.1)  # Variance control

    def forward(self, x):
        diff = x.unsqueeze(1) - self.centers.unsqueeze(0)  # Compute distances
        return torch.exp(-self.beta * torch.sum(diff ** 2, dim=2))  # Gaussian function

# RBF Network
class RBF_NN(nn.Module):
    def __init__(self, input_dim, num_centers, output_dim):
        super().__init__()
        self.rbf_layer = RBF_Layer(num_centers, input_dim)
        self.output_layer = nn.Linear(num_centers, output_dim)

    def forward(self, x):
        rbf_out = self.rbf_layer(x)
        return self.output_layer(rbf_out)

# Example usage
input_dim = 2  # (Δx, Δy)
num_centers = 10  # Number of Gaussian nodes
output_dim = 1  # Predict stress

model = RBF_NN(input_dim, num_centers, output_dim)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training data
x_train = torch.randn(100, 2)
y_train = torch.randn(100, 1)

# Forward pass
y_pred = model(x_train)


In [2]:
# Gaussian Mixture Model

import torch
import torch.nn.functional as F

class GMM_NN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_gaussians):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_gaussians * 3)  # Mixture: Mean, Variance, Weight
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        params = self.fc2(x)
        means, log_vars, weights = torch.chunk(params, 3, dim=1)
        weights = F.softmax(weights, dim=1)
        return means, torch.exp(log_vars), weights  # Ensure variances are positive

# Example usage
input_dim = 2
hidden_dim = 32
num_gaussians = 3

model = GMM_NN(input_dim, hidden_dim, num_gaussians)
x_train = torch.randn(100, 2)
means, variances, weights = model(x_train)


NameError: name 'nn' is not defined

In [ ]:
# Bayesian NN

import torchbnn as bnn

class BayesianNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features=input_dim, out_features=hidden_dim)
        self.fc2 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features=hidden_dim, out_features=output_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Example usage
input_dim = 2
hidden_dim = 32
output_dim = 1

model = BayesianNN(input_dim, hidden_dim, output_dim)
